# 00 - Workshop Setup & Configuration

**Azure ML MLOps Workshop**

This notebook sets up the connection to your Azure ML workspace and verifies all prerequisites.

### What you'll do:
1. Install required packages
2. Connect to Azure ML workspace
3. Verify compute resources
4. Verify data access

---

## 1. Install Dependencies

Run this if you're on a fresh compute instance.

In [ ]:
%pip install -r ../requirements.txt -q

## 2. Connect to Azure ML Workspace

Fill in your workspace details below. If you're running on an Azure ML compute instance,
the `DefaultAzureCredential` will automatically pick up your identity.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# ============================================================
# UPDATE THESE VALUES FOR YOUR ENVIRONMENT
# ============================================================
SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)
# ============================================================

credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

workspace = ml_client.workspaces.get(WORKSPACE_NAME)
print(f"Connected to workspace: {workspace.name}")
print(f"  Location: {workspace.location}")
print(f"  Resource group: {workspace.resource_group}")

## 3. Verify Compute Resources

We need:
- A **compute instance** (for running these notebooks)
- A **compute cluster** (for training jobs)

In [ ]:
from azure.ai.ml.entities import AmlCompute

# List existing compute
print("Existing compute targets:")
for compute in ml_client.compute.list():
    print(f"  {compute.name} ({compute.type}) - {compute.size}")

# Create a compute cluster if it doesn't exist
CLUSTER_NAME = "cpu-cluster"

try:
    cluster = ml_client.compute.get(CLUSTER_NAME)
    print(f"\nCompute cluster '{CLUSTER_NAME}' already exists.")
except Exception:
    print(f"\nCreating compute cluster '{CLUSTER_NAME}'...")
    cluster = AmlCompute(
        name=CLUSTER_NAME,
        type="amlcompute",
        size="Standard_DS3_v2",
        min_instances=0,
        max_instances=2,
        idle_time_before_scale_down=300,
    )
    ml_client.compute.begin_create_or_update(cluster).result()
    print(f"Compute cluster '{CLUSTER_NAME}' created.")

## 4. Quick Sanity Check - Read the Data

Verify we can access the datasets from this environment.

In [ ]:
import pandas as pd

DATA_DIR = "../data"

df_inspections = pd.read_csv(f"{DATA_DIR}/inspections_dataset.csv")
df_os = pd.read_csv(f"{DATA_DIR}/service_orders_dataset.csv")

print(f"Inspections dataset: {df_inspections.shape[0]:,} rows, {df_inspections.shape[1]} columns")
print(f"Service orders dataset: {df_os.shape[0]:,} rows, {df_os.shape[1]} columns")
print()
print("Inspections columns:", list(df_inspections.columns))
print("Service orders columns:", list(df_os.columns))

In [ ]:
df_inspections.head()

In [ ]:
df_os.head()

## Setup Complete!

You're ready to proceed to **Notebook 01 - Data Versioning**.

- **Track A (Text):** [01 - Data Versioning](./track_a_text/01_data_versioning.ipynb)
- **Track B (Tabular):** [01b - Data Versioning](./track_b_tabular/01b_data_versioning_tabular.ipynb)

### Quick Summary

| Component | Status |
|-----------|--------|
| Azure ML Workspace | Connected |
| Compute Cluster | Ready |
| Inspections Dataset | 10,500 rows - Text classification (sales lead detection) |
| Service Orders Dataset | 425,745 rows - Tabular classification (equipment maintenance) |

**The ML problem:** Given an inspection comment, predict whether it represents a sales lead opportunity (`is_lead_opportunity`).